In [ ]:
pip install datasets

In [3]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

In [4]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import pandas as pd

df_train_sampled = pd.read_csv('train_sampled.csv')
df_test_sampled = pd.read_csv('test_sampled.csv')

In [6]:
train_encodings = tokenizer(list(df_train_sampled['text']), truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(list(df_test_sampled['text']), truncation=True, padding=True, max_length=256)

In [7]:
train_dataset = Dataset.from_dict(train_encodings)
test_dataset = Dataset.from_dict(test_encodings)

In [8]:
train_dataset = train_dataset.add_column('label', df_train_sampled['label'].values)
test_dataset = test_dataset.add_column('label', df_test_sampled['label'].values)

In [25]:
training_args = TrainingArguments(
    output_dir='./results',          # output directory for model checkpoints
    num_train_epochs=20,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=64,   # batch size for evaluation
    evaluation_strategy="epoch",     # evaluate after each epoch
    save_strategy="epoch",           # save after each epoch
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,                # how often to log
    load_best_model_at_end=True,     # load the best model at the end of training
    metric_for_best_model="f1",     # the metric to monitor for selecting the best model
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [26]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(p):
    predictions, labels = p
    # Convert predictions to tensor if they are a numpy array
    predictions = torch.tensor(predictions)

    # Get the predicted class labels (argmax returns the index of the maximum value along the specified axis)
    preds = torch.argmax(predictions, axis=1)

    # Calculate precision, recall, f1 score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

    # Calculate accuracy
    accuracy = accuracy_score(labels, preds)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [27]:
trainer = Trainer(
    model=model,                         # the model to train
    args=training_args,                  # training arguments
    train_dataset=train_dataset,         # training dataset
    eval_dataset=test_dataset,           # evaluation dataset
    compute_metrics=compute_metrics      # evaluation metrics
)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.415100,0.597240,0.720000,0.702206,0.764000,0.731801
2,0.430300,0.605517,0.727500,0.743316,0.695000,0.718346
3,0.280900,0.708672,0.732000,0.733400,0.729000,0.731194


In [ ]:
results = trainer.evaluate()
print("Evaluation Results:", results)

Epoch	Training Loss	Validation Loss	Accuracy	Precision	Recall	F1
1	0.511800	0.515474	0.734500	0.696893	0.830000	0.757645